In [ ]:
from transformer_lens import HookedTransformer

model: HookedTransformer = HookedTransformer.from_pretrained("pythia-14M")

logits, cache = model.run_with_cache("The sky is bluue and the grass is green.")

print(logits.shape)  # ty:ignore[unresolved-attribute]

for k, v in cache.items():
    print(k, v.shape)

# d_vocab = 50304
# d_model = 128
# n_head = 4,
# d_head = 32
# d_mlp = 512

Loaded pretrained model pythia-14M into HookedTransformer
torch.Size([1, 13, 50304])
hook_embed torch.Size([1, 13, 128])
blocks.0.hook_resid_pre torch.Size([1, 13, 128])
blocks.0.ln1.hook_scale torch.Size([1, 13, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 13, 128])
blocks.0.attn.hook_q torch.Size([1, 13, 4, 32])
blocks.0.attn.hook_k torch.Size([1, 13, 4, 32])
blocks.0.attn.hook_v torch.Size([1, 13, 4, 32])
blocks.0.attn.hook_rot_q torch.Size([1, 13, 4, 32])
blocks.0.attn.hook_rot_k torch.Size([1, 13, 4, 32])
blocks.0.attn.hook_attn_scores torch.Size([1, 4, 13, 13])
blocks.0.attn.hook_pattern torch.Size([1, 4, 13, 13])
blocks.0.attn.hook_z torch.Size([1, 13, 4, 32])
blocks.0.hook_attn_out torch.Size([1, 13, 128])
blocks.0.ln2.hook_scale torch.Size([1, 13, 1])
blocks.0.ln2.hook_normalized torch.Size([1, 13, 128])
blocks.0.mlp.hook_pre torch.Size([1, 13, 512])
blocks.0.mlp.hook_post torch.Size([1, 13, 512])
blocks.0.hook_mlp_out torch.Size([1, 13, 128])
blocks.0.hook_resid_post torch

In [6]:
for x in model.state_dict().keys():
    print(x)
print(model.cfg)

embed.W_E
blocks.0.attn.W_Q
blocks.0.attn.W_O
blocks.0.attn.b_Q
blocks.0.attn.b_O
blocks.0.attn.W_K
blocks.0.attn.W_V
blocks.0.attn.b_K
blocks.0.attn.b_V
blocks.0.attn.mask
blocks.0.attn.IGNORE
blocks.0.attn.rotary_sin
blocks.0.attn.rotary_cos
blocks.0.mlp.W_in
blocks.0.mlp.b_in
blocks.0.mlp.W_out
blocks.0.mlp.b_out
blocks.1.attn.W_Q
blocks.1.attn.W_O
blocks.1.attn.b_Q
blocks.1.attn.b_O
blocks.1.attn.W_K
blocks.1.attn.W_V
blocks.1.attn.b_K
blocks.1.attn.b_V
blocks.1.attn.mask
blocks.1.attn.IGNORE
blocks.1.attn.rotary_sin
blocks.1.attn.rotary_cos
blocks.1.mlp.W_in
blocks.1.mlp.b_in
blocks.1.mlp.W_out
blocks.1.mlp.b_out
blocks.2.attn.W_Q
blocks.2.attn.W_O
blocks.2.attn.b_Q
blocks.2.attn.b_O
blocks.2.attn.W_K
blocks.2.attn.W_V
blocks.2.attn.b_K
blocks.2.attn.b_V
blocks.2.attn.mask
blocks.2.attn.IGNORE
blocks.2.attn.rotary_sin
blocks.2.attn.rotary_cos
blocks.2.mlp.W_in
blocks.2.mlp.b_in
blocks.2.mlp.W_out
blocks.2.mlp.b_out
blocks.3.attn.W_Q
blocks.3.attn.W_O
blocks.3.attn.b_Q
blocks.3.att

In [7]:
from transformer_lens import HookedTransformer


def logit_lens(model: HookedTransformer, prompt: str):
    """
    Run logit lens on a prompt — projects residual stream at each layer
    through the final LayerNorm + unembedding to get token predictions.
    """
    tokens = model.to_tokens(prompt)
    str_tokens = model.to_str_tokens(prompt)
    n_layers = model.cfg.n_layers

    # Cache all residual stream states
    _, cache = model.run_with_cache(
        tokens, names_filter=lambda name: name.endswith("hook_resid_post")
    )

    results = []

    for layer in range(n_layers):
        resid = cache[f"blocks.{layer}.hook_resid_post"]  # (1, seq_len, d_model)

        # Apply final layer norm + unembed (the "logit lens" projection)
        logits = model.unembed(model.ln_final(resid))  # (1, seq_len, vocab_size)
        top_tokens = logits[0].argmax(dim=-1)  # (seq_len,)
        top_str_tokens = [model.tokenizer.decode(t) for t in top_tokens]

        results.append((layer, top_str_tokens))

    return str_tokens, results


# ── Example usage ──────────────────────────────────────────────────────────────
prompt = "The Eiffel Tower is located in the city of"
str_tokens, results = logit_lens(model, prompt)

print(f"Input tokens: {str_tokens}\n")
print(f"{'Layer':<8}", end="")
for tok in str_tokens:
    print(f"{repr(tok):>15}", end="")
print()

for layer, preds in results:
    print(f"L{layer:<7}", end="")
    for pred in preds:
        print(f"{repr(pred):>15}", end="")
    print()

Input tokens: ['<|endoftext|>', 'The', ' E', 'iff', 'el', ' Tower', ' is', ' located', ' in', ' the', ' city', ' of']

Layer   '<|endoftext|>'          'The'           ' E'          'iff'           'el'       ' Tower'          ' is'     ' located'          ' in'         ' the'        ' city'          ' of'
L0                 '"('          'ODU'            '.'            ','            '-'            ','           'од'            '.'            '/'           '@"'            ','            '.'
L1                 '"('       '\n\t\t'            '9'            '.'         ' and'            ','         'itle'        ' more'            '-'           '}"'            ','            '.'
L2                 ' "'            '-'            '-'            '-'            '-'           'ed'       ' being'           'ég'         ' the'           ' "'           "'s"         ' and'
L3                 ' "'           'if'            '-'            '-'            '-'            '-'           ' "'          'd

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.functional import kl_div, log_softmax, softmax

# ── Model ──────────────────────────────────────────────────────────────────────
model = HookedTransformer.from_pretrained("pythia-14m")
model.eval()


# ── Tuned Lens: one affine translator T_l per layer ───────────────────────────
class TunedLens(nn.Module):
    def __init__(self, n_layers: int, d_model: int):
        super().__init__()
        # Each translator is a learned affine map: x -> A_l @ x + b_l
        # Initialised to identity so it starts identical to logit lens
        self.translators = nn.ModuleList(
            [nn.Linear(d_model, d_model, bias=True) for _ in range(n_layers)]
        )
        for t in self.translators:
            nn.init.eye_(t.weight)
            nn.init.zeros_(t.bias)

    def forward(self, resid: torch.Tensor, layer: int) -> torch.Tensor:
        return self.translators[layer](resid)


def decode(model: HookedTransformer, resid: torch.Tensor) -> torch.Tensor:
    """Apply final LN + unembed to get logits. Works on any resid shape."""
    return model.unembed(model.ln_final(resid))


# ── Training ───────────────────────────────────────────────────────────────────
def train_tuned_lens(
    model: HookedTransformer,
    prompts: list[str],
    n_epochs: int = 50,
    lr: float = 1e-3,
) -> TunedLens:
    """
    Train one affine translator per layer to minimise KL(tuned_lens || final_layer).
    """
    n_layers = model.cfg.n_layers
    d_model = model.cfg.d_model
    lens = TunedLens(n_layers, d_model).to(model.cfg.device)
    optimizer = optim.Adam(lens.parameters(), lr=lr)

    hook_names = [f"blocks.{l}.hook_resid_post" for l in range(n_layers)]

    for epoch in range(n_epochs):
        total_loss = 0.0

        for prompt in prompts:
            tokens = model.to_tokens(prompt)

            with torch.no_grad():
                final_logits, cache = model.run_with_cache(
                    tokens, names_filter=lambda n: n in hook_names
                )
                # Target: softmax distribution from the final layer
                target_probs = softmax(final_logits, dim=-1)  # (1, seq, vocab)

            optimizer.zero_grad()
            loss = torch.tensor(0.0, device=model.cfg.device, requires_grad=True)

            for layer in range(n_layers):
                resid = cache[hook_names[layer]]  # (1, seq, d_model)
                translated = lens(resid, layer)  # (1, seq, d_model)
                logits_l = decode(model, translated)  # (1, seq, vocab)

                # KL(final || tuned_lens) — train lens to match final layer
                log_pred = log_softmax(logits_l, dim=-1)
                kl = kl_div(
                    log_pred, target_probs, reduction="batchmean", log_target=False
                )
                loss = loss + kl

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch:>3} | loss: {total_loss / len(prompts):.4f}")

    return lens


# ── Inference ──────────────────────────────────────────────────────────────────
def run_tuned_lens(
    model: HookedTransformer,
    lens: TunedLens,
    prompt: str,
) -> tuple[list[str], list[tuple[int, list[str]]]]:
    """Project each layer's residual stream through the tuned lens and decode."""
    tokens = model.to_tokens(prompt)
    str_tokens = model.to_str_tokens(prompt)
    n_layers = model.cfg.n_layers
    hook_names = [f"blocks.{l}.hook_resid_post" for l in range(n_layers)]

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda n: n in hook_names)

    results = []
    for layer in range(n_layers):
        resid = cache[hook_names[layer]]
        translated = lens(resid, layer)
        logits_l = decode(model, translated)
        top_tokens = logits_l[0].argmax(dim=-1)
        preds = [model.tokenizer.decode(t) for t in top_tokens]
        results.append((layer, preds))

    return str_tokens, results


# ── Example ────────────────────────────────────────────────────────────────────
training_prompts = [
    "The Eiffel Tower is located in the city of",
    "The capital of Germany is the city of",
    "Water is made of hydrogen and",
    "The president of the United States is",
    "To be or not to be, that is the",
]

print("Training tuned lens...")
lens = train_tuned_lens(model, training_prompts, n_epochs=50, lr=1e-3)


# ── Compare logit lens vs tuned lens side by side ─────────────────────────────
def compare_lenses(model, lens, prompt):
    tokens = model.to_tokens(prompt)
    str_tokens = model.to_str_tokens(prompt)
    n_layers = model.cfg.n_layers
    hook_names = [f"blocks.{l}.hook_resid_post" for l in range(n_layers)]

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda n: n in hook_names)

    print(f"\nPrompt: {repr(prompt)}")
    print(f"Tokens: {str_tokens}\n")
    print(f"{'Layer':<6} {'Lens':<12}", end="")
    for tok in str_tokens:
        print(f"{repr(tok):>14}", end="")
    print()
    print("-" * (18 + 14 * len(str_tokens)))

    for layer in range(n_layers):
        resid = cache[hook_names[layer]]

        # Logit lens (no translator)
        logit_preds = decode(model, resid)[0].argmax(-1)
        logit_strs = [model.tokenizer.decode(t) for t in logit_preds]

        # Tuned lens (with translator)
        with torch.no_grad():
            tuned_preds = decode(model, lens(resid, layer))[0].argmax(-1)
        tuned_strs = [model.tokenizer.decode(t) for t in tuned_preds]

        print(f"L{layer:<5} {'logit':<12}", end="")
        for p in logit_strs:
            print(f"{repr(p):>14}", end="")
        print()
        print(f"{'':6} {'tuned':<12}", end="")
        for p in tuned_strs:
            print(f"{repr(p):>14}", end="")
        print()


compare_lenses(model, lens, "The Eiffel Tower is located in the city of")

Loaded pretrained model pythia-14m into HookedTransformer
Training tuned lens...
Epoch   0 | loss: 261.2361
Epoch  10 | loss: 74.6144
Epoch  20 | loss: 44.6610
Epoch  30 | loss: 31.3973
Epoch  40 | loss: 24.5650

Prompt: 'The Eiffel Tower is located in the city of'
Tokens: ['<|endoftext|>', 'The', ' E', 'iff', 'el', ' Tower', ' is', ' located', ' in', ' the', ' city', ' of']

Layer  Lens        '<|endoftext|>'         'The'          ' E'         'iff'          'el'      ' Tower'         ' is'    ' located'         ' in'        ' the'       ' city'         ' of'
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
L0     logit                 '"('         'ODU'           '.'           ','           '-'           ','          'од'           '.'           '/'          '@"'           ','           '.'
       tuned                  'Q'    ' present'          